In [116]:
import pandas as pd 
import numpy as np


In [117]:
from email_validator import validate_email, EmailNotValidError

In [118]:
#DataFrame
df = pd.read_csv('D:\python\Python_Data_Cleanning_Project\Data\super_dirty_students.csv')


<>:2: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<>:2: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
C:\Users\User\AppData\Local\Temp\ipykernel_8692\1716561489.py:2: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
  df = pd.read_csv('D:\python\Python_Data_Cleanning_Project\Data\super_dirty_students.csv')


In [119]:
#Column 1 -- student_id
#Find out how many unique values
df.student_id.unique()
#Student_id has 1000 unique valuse.It means everything is fine.

#Finding duplicates 
filter = df.student_id.duplicated() 
df.student_id.loc[filter]
# It did not show anything it means student_=id column does not have any duplicate values.

#Finding its datatype 
df.student_id.dtype
#Student_id data type is int64 so we should not change it.


dtype('int64')

In [120]:
#Second Column -- name
df.name.describe()
#Thi column has 665 unique names and othars are nulls. 

df['name'] = df['name'].str.strip()
#Finding null values

filter_name = df.name.isna()

df['name'].loc[filter_name]

# We have 335 missin(null) values in name column. we do not change anything with null values.

#Standardization : Making capitalize all first charachters of all words.

df['name'] = df['name'].str.title()


In [ ]:
#Third Column: age 

df.age.info()
# age column has 892 non-null values. 
#Finding any unvalid formats
df.age.unique()
#we have 'twenty' and '18 years' formats. we will convert them into numbers

df['age'] = df['age'].replace('twenty',20).replace('18 years',18).astype('float64')

#we will change null values with average age
avg_age = df.age.mean()
#Change datatype from float to integer
df['age'] = df['age'].fillna(avg_age).astype('int64')
df

In [122]:
#Fourth Column. Gender
df.gender.describe()
#we have 8 various types of gender. we should make them three('Male','|Female','Unknown')
df.gender.unique()

#bY Making them capitalize, we elimnated 3 types of genders. 
df['gender'] = df['gender'].str.title()

# we replace unvalid types with valid ones.
df['gender'] = df['gender'].replace('Fmale','Female').replace('Femlae','Female').fillna('Unknown')
df.gender.unique()
#After replacing all unvalid gendertypes, we have standard three gender types.
# We will check again for Null valuse. 
filter_gender = df.gender.isnull()
df.gender.loc[filter_gender]


Series([], Name: gender, dtype: str)

In [ ]:
#Fifth Column: score
df.score.describe()
#we have 754 values out of 1000 and others are null values. we will replace null values with average score.
#Finding unvalid values. we have two(nan,ninety), and replaceing them 
df.score.unique()
df['score'] = df['score'].replace('ninety',90).astype('float32').fillna(df.score.mean()).astype('int64')
df

In [126]:
#Sixth Column: Phone Number
df['phone'] = df['phone'].astype(str).str.strip().str.split('x').str[0].str.replace('+','').str.replace(r'^001','',regex=True).str.replace(r'\)|\(|\-|\.','',regex=True)
df['phone'] = '+'+df['phone']



In [ ]:
#7-column:City
df.city.describe()
df.city.info()
# In city columns, we do not have any duplicates. so we just remove trial and lead spaces after that make capitalize each word's first charchter
df['city'] = df['city'].str.strip().str.title()
df.city.head(30)

In [ ]:
#8-Column: Email
df.email.info()
#We have 849 non-null values out of 1000 values.
df.email.describe()
#We have 567 unique email values.
filter = df.email.duplicated()
df.email.loc[filter]
#we have a lot of duplicate emails. 
df['email'] = df['email'].str.strip().str.replace('@@','@').str.replace(r'(?<!@)gmail', r'@gmail', regex=True).str.replace(r'^[^a-zA-Z0-9]+', '', regex=True)

email_regex = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'

df['email'] = df['email'].where(df['email'].str.contains(email_regex, regex=True, na=False), None)

def check_email(e):
    if pd.isna(e) or e is None: 
        return False
    try:
        # check_deliverability=False focuses strictly on complex syntax checks
        validate_email(str(e), check_deliverability=False)
        return True
    except (EmailNotValidError, ValueError, TypeError):
        return False

# Generate the true/false map for every row
is_valid_syntax = [check_email(email) for email in df['email']]

# Update only the column (keeps valid emails, turns invalid/fake ones into None)
df['email'] = df['email'].where(is_valid_syntax, None)


df['email'].head(30)


In [ ]:
#9-column: date_of_join
df.date_of_join.head(30)

df['date_of_join'] = pd.to_datetime(df['date_of_join'],errors='coerce')
df.date_of_join.head(30)

In [ ]:
#10-column: course
df.course.unique()

values_to_replace = {
    'DATA SCIENCE':'Data Science',
    'data-sciens':'Data Science',
    'data_sciense':'Data Science',
    'data science':'Data Science',
    'python':'Python',
    'ds':'Data Science',
    'd.s.':'Data Science',
    'PYTHNO':'Python',
    'Pyhton':'Python'
}

df['course'] = df['course'].str.replace(values_to_replace)
df.course.head(30)

In [ ]:
#11-column: attendance
df.attendance.unique()
#Replacing % symbols with spaces
df['attendance'] = df['attendance'].str.replace('%','')
#Changing data type from str and filling null values with averge attendance and changing data type from float64 to int64
df['attendance'] = df['attendance'].astype('float64').fillna(df.attendance.mean()).astype('int64')
df.attendance.unique()


In [ ]:
#12-column:status
df.status.unique()
#All the values are valid so just remove extra space

df['status'] = df['status'].str.strip().str.title()
df.status.unique()

In [ ]:
#13-column:gpa
df.gpa.unique()
#We will find unique values with letters
df.loc[df['gpa'].str.contains(r'[a-zA-Z]',na=False),'gpa'].unique()
#we found only one value with letter:four point five

df['gpa'] = df['gpa'].replace({'four point five':4.5,'3,7':3.7}).astype('float64').abs().fillna(df.gpa.mean()).round(2)
df.gpa

In [143]:
#14-column:remarks
df.remarks.unique()
df['remarks'] = df['remarks'].str.strip().str.title()
df.remarks.unique()

<StringArray>
['Good', 'Excellent', 'Average', nan]
Length: 4, dtype: str

In [ ]:
#15-column"money_spent
df.money_spent.unique()

df['money_spent'] = df['money_spent'].str.replace({'USD':'','$':'',',':'.'})

df['money_spent'] = df['money_spent'].astype('float64').fillna(df.money_spent.mean()).round(2)




In [ ]:
#16-column:event_time
df['event_time'] = pd.to_datetime(df['event_time'],errors='coerce')

df.event_time.head(30)

In [ ]:
#17-column:address_raw 
import re 
df.address_raw.info()
df.address_raw.describe()
#Changeing borken address values to NaN
df['address_raw'] = df['address_raw'].replace('BROKEN,ADDRESS,DATA,,,',np.nan)
df.address_raw.describe()


import re 


def parse_addres(adr):
    if not isinstance(adr,str) or "BROKEN" in adr.upper():
        return pd.Series([np.nan,np.nan,np.nan])
    
    city=np.nan
    district=np.nan
    postal=np.nan

    postal_match=re.search(r'\b(?:UZ\s*)?(\d{5,6})\b',adr,re.IGNORECASE)
    if postal_match:
        postal=postal_match.group(1)

    if "tashkent" in adr.lower():
        city='Tashkent'
    else:
        parts=[p.strip() for p in adr.split(",")]  
        if parts:
            last_part=parts[-1]  
            if re.search(r'^\d+$',last_part) and len(parts)>1:
                last_part=parts[-2]
            clean_city=re.sub(r"\b(?:UZ\s*)?\d{5,6}\b", "", last_part).strip()
            if clean_city:
                city=clean_city

    district_match=re.search(r'\b([\w\s\-]+district)\b',adr,re.IGNORECASE)

    if district_match:
        district=district_match.group(1).strip()

    elif "kv" in adr.lower():
        kv_match=re.search(r"\b([\w\s\-]+\d+\-kv)\b", adr, re.IGNORECASE)
        if kv_match:
            district=kv_match.group(1).strip()
    return pd.Series([city,district,postal])
        
df[['adr_city','adr_district','adr_postal']]=df['address_raw'].apply(parse_addres)

df[['adr_city','adr_district','adr_postal']]



In [ ]:
#18-column:profile_json

import json
import re



# {hobbies: [...]} → {"hobbies": [...]} formatga o'tkazish
def parse(x):
    if pd.isna(x) or str(x).strip() in ("", "null", "None"):
        return {}
    text = str(x).strip()
    # quotes yo'q key larni fix qilish: {key: → {"key":
    text = re.sub(r"(?<=[{,])\s*([a-zA-Z_][a-zA-Z0-9_]*)\s*:", r'"\1":', text)
    # single quote → double quote
    text = text.replace("'", '"')
    try:
        return json.loads(text)
    except Exception:
        return {}

parsed = df["profile_json"].apply(parse)

# hobbies — list bo'lsa vergul bilan
df["hobbies"] = parsed.apply(
    lambda x: ", ".join(str(i) for i in x["hobbies"])
    if isinstance(x.get("hobbies"), list) else x.get("hobbies")
)

# skills — nested dict/list bo'lishi mumkin, to'liq flatten
df["skills"] = parsed.apply(
    lambda x: " | ".join(
        f"{sk}: {', '.join(str(v) for v in sv)}" if isinstance(sv, list)
        else f"{sk}: {', '.join(f'{k}={v}' for k, v in sv.items())}" if isinstance(sv, dict)
        else f"{sk}: {sv}"
        for sk, sv in x["skills"].items()
    ) if isinstance(x.get("skills"), dict)
    else ", ".join(str(i) for i in x["skills"]) if isinstance(x.get("skills"), list)
    else x.get("skills")
)

# family — nested dict flatten
df["family"] = parsed.apply(
    lambda x: " | ".join(
        f"{k}: {', '.join(f'{ik}={iv}' for ik, iv in v.items())}" if isinstance(v, dict)
        else f"{k}: {v}"
        for k, v in x["family"].items()
    ) if isinstance(x.get("family"), dict) else x.get("family")
)

# devices — list of dict flatten
df["devices"] = parsed.apply(
    lambda x: "; ".join(
        " | ".join(f"{k}: {v}" for k, v in d.items()) if isinstance(d, dict) else str(d)
        for d in x["devices"]
    ) if isinstance(x.get("devices"), list) else x.get("devices")
)

df[["hobbies", "skills", "family", "devices"]].head(5).to_string()


In [ ]:
#writing to new csv file 
df.to_csv("students_cleaned.csv", index=False)